# ATP uncaging wave analysis, `(I - min)/min` normalization

Every sector's time course is normalized per sector as `(I - min_over_time) / min_over_time`, so each region
(cell) is scaled by its own GCaMP baseline.

**Input.** One AVI per stack (see README.md). Each AVI frame is the average of 10 raw frames of 0.7 s, so **one AVI frame = 7 s** (40 frames = 280 s); 1 px = 1 µm. The frame rate written in the AVI header is not used: frames are read by index and converted with `FRAME_INTERVAL_S`. The uncaging light is on from frame 0.

**Groups.** All stacks are run and pooled by **condition × uncage time**:

| Group | Stacks | Uncaging circle |
|---|---|---|
| ATP, uncage full | ATP1_2 | on until raw frame 400/400 (whole movie, 280 s) |
| ATP, uncage 35 s | ATP3, ATP4, ATP5 | off after raw frame ~50/400 (35 s, AVG frame 5) |
| suramin, uncage full | Suramin1, suramin 2, suramin 3 | on until raw frame 400/400 |

ATP4 is also in the `uncage time (full)` folder (identical file). Its circle disappears at AVG frame 5, as in ATP3
and ATP5, so it is counted once, in the 35 s group. ATP2 is not included.

**Sheet wave only, full circle.** Besides the wave, single cells flash on their own throughout the movie (they
flash just as much under suramin, where there is no wave). Each sector is therefore measured on the pixels
reached by the sheet of responding cells that grows out from the lit circle, with the same code as the wound
analysis (`ca_wave/sheet_wave.py`); pixels that flash on their own are left out. The full circle is measured
(all 32 wedges, tissue pixels only), except in suramin 2 and 3, where the left half is measured to keep the wound
on the right of the image out.

Paths: run this notebook from the `notebooks/` folder. Data and results folders default to `data/` and `results/` in the repository and can be changed with the environment variables listed in README.md.


## Using your own data: checklist

There are no example uncaging movies in this repository; the movies of the paper are listed in README.md.

1. **Movies.** One AVI per uncaging experiment, single-channel (grayscale), one analysed time point per frame.
   Frames are read by index; the frame rate in the AVI header is not used. Put the movies anywhere under
   `data/uncaging/` (or point `UNCAGING_DATA_DIR` to where they are).
2. **`STACKS`** (section "Settings"): one entry per movie with
   - `name`: label and output folder;
   - `condition` and `uncage_time`: movies are pooled by `condition, uncage <uncage_time>` (any text);
   - `path`: the AVI path relative to `data/uncaging/` (if it is not found there, the file is searched for by
     name anywhere under `data/uncaging/`);
   - `angles`: the part of the circle to measure, `(lo, hi)` in degrees, `0°` = 3 o'clock, increasing clockwise;
     `(0, 360)` = the full circle; use a smaller range only to keep out another injury in the field (figure 2
     shows the full circle);
   - `uncage_end_raw`: the raw frame in which the uncaging light goes off (only drawn as a dashed line in the
     figures; converted with `RAW_FRAMES_PER_FRAME`);
   - `circle`: `None` to detect the lit circle automatically, or `(cx, cy, r)` in pixels if detection fails.
3. **Acquisition.** `PIXEL_SIZE_UM` (µm per pixel) and `FRAME_INTERVAL_S` (seconds per AVI frame); set
   `RAW_FRAMES_PER_FRAME` if each AVI frame averages a different number of raw frames.
4. **The lit circle** is detected on the per-pixel minimum of `DETECT_FRAMES` (frames 0 and 1), so the light
   must be on from frame 0 and the lit area must be brighter than its surroundings. `RADIUS_RANGE` sets the
   radii searched; `SEARCH_BOX` can restrict where the centre may lie. Check the circle in figure 1.
5. **If your pixel size is not 1 µm:** the rings are defined in pixels. To keep the paper's 25-µm rings out to
   200 µm, set `RING_STEP = round(25 / PIXEL_SIZE_UM)`, `RING_MAX_OFFSET = 8 * RING_STEP` and
   `FIT_MIN_OFFSET = RING_STEP`.
6. **Run all cells** and read each movie's verdict (section "How to read the results and what to adjust").

The measurement and wave-test settings are those of the paper; keep them to compare your data with the paper's.

**Method choices.**

- **Arrival time per ring.** Each sector's arrival is its own half-max crossing, so a fit can come out flat or
  negative; no path is forced to be monotonic in time.
- **Edge ring not used for timing.** The ring on the circle edge (offset 0) is never used for timing
  (`FIT_MIN_OFFSET = 25`): in suramin it responds late and would bias the timing.
- **One fixed colour scale** (`NORM_VMAX`) for every heatmap of every stack, so small fluctuations are not drawn as
  large responses.
- **A wave must be shown to exist.** A sector must respond (normalized peak ≥ `MIN_PEAK` and ≥ `NOISE_K` × its
  noise). A wave must reach ≥ `MIN_WAVE_RINGS` rings and arrive later at larger distances (ring-label permutation
  test, p < `ALPHA`). Otherwise the stack is reported as **no outward wave** and no speed is given.
- **No outlier removal.** The median arrival per ring is used.
- **Activity entering from the image edge** arrives *earlier* far from the circle, so it fails the outward-order
  test. The full-circle maps show where it comes from.
- **Ring bands, not thin arcs.** Each sector averages its whole 25-px ring band.
- **Responses must rise after the start.** With `(I - min)/min`, a trace that starts high and then dims (slow
  bleaching, or a cell already bright when the movie starts) has its peak at frame 0. A sector only responds if its
  trace **starts below half of its peak and rises** (`starts_low`). A real uncaging response cannot arrive at
  frame 0 (the wave reaches 25 µm at about frame 3 in every ATP stack).
- **Sheet pixels only.** Each sector's trace is the mean of its sheet pixels, the pixels reached by the sheet that
  grows out from the lit circle while the light is on (section "Measurement and wave test"). Cells that flash on
  their own, far from the sheet, do not set arrival times.
- **Immediate response and spreading wave.** The rings reached within 21 s (`INITIAL_WINDOW_S`) are the
  immediate response. An outward wave must reach at least 2 rings (`MIN_BEYOND_RINGS`, 50 µm) beyond it, so a
  response that only covers a region at once is not called a wave.

Angles: `0°` = 3 o'clock, increasing clockwise (`90°` = 6 o'clock, `180°` = 9 o'clock, `270°` = 12 o'clock).

In [ ]:
import json
import os
import warnings
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Wedge
from scipy.signal import fftconvolve
from scipy.ndimage import map_coordinates
from scipy.stats import theilslopes, kendalltau
from skimage.filters import gaussian
from skimage.transform import downscale_local_mean
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

import sys
sys.path.insert(0, str(Path.cwd().parent))   # the repository root: the shared ca_wave package
from ca_wave.sheet_wave import SHEET_SETTINGS, sheet_pixels


## Settings

Add one row to `STACKS` per movie and set its `uncage_time` (`"full"` or `"35 s"`); results are pooled by
`group` = condition + uncage time. `angles` is the part of the circle that is measured: `(0, 360)` = the full circle (every stack except
suramin 2 and 3, where `(90, 270)`, the left half, keeps the wound on the right of the image out). Check figure 2
(full-circle maps) and figure 5 (sheet pixels).

In [ ]:
# ---- paths (see README.md) ----
REPO_ROOT = Path.cwd().parent   # this notebook runs from <repository>/notebooks
DATA_ROOT = Path(os.environ.get("UNCAGING_DATA_DIR", REPO_ROOT / "data" / "uncaging"))   # the AVI movies

# every sector is normalized as (I - min over time) / min
NORM_LABEL = "(I-min)/min"
OUT_ROOT = Path(os.environ.get("RESULTS_DIR", REPO_ROOT / "results")) / "uncaging_wave_analysis"

#   name           : label for tables/figures and the output subfolder
#   condition      : "ATP" or "suramin"
#   uncage_time    : "full" (circle on to raw frame 400/400) or "35 s" (circle off after raw frame 50/400)
#                    results are pooled by group = condition + uncage time
#   path           : AVI path relative to DATA_ROOT
#   angles         : measured angle range (lo, hi) in degrees; hi <= lo wraps through 0; (0, 360) = full circle.
#                    The full circle is measured, except where another injury is in the field (suramin 2 and 3:
#                    a wound on the right, so only the left half is measured)
#   uncage_end_raw : raw frame (of 400) when the uncaging circle disappears (dashed line on the kymographs)
#   circle         : None = detect automatically, or (cx, cy, r) in px to override a wrong detection
ATP_FULL = "101626 ATP uncaging (AVG_Stack)/uncage time (full)"
ATP_35S = "101626 ATP uncaging (AVG_Stack)/uncage time (35s or frame 50-400)"
SURAMIN = "100126 uncaging suramin (AVG_Stack)"
STACKS = [
    dict(name="ATP1_2", condition="ATP", uncage_time="full",
         path=f"{ATP_FULL}/AVG_Stack_ATP1_2.avi", angles=(0, 360), uncage_end_raw=400, circle=None),
    # ATP4: the same file is also in the "uncage time (full)" folder, but its circle turns off at ~frame 50/400
    # (edge contrast drops at AVG frame 5), so it is counted once, in the 35 s group
    dict(name="ATP3", condition="ATP", uncage_time="35 s",
         path=f"{ATP_35S}/AVG_Stack_ATP3.avi", angles=(0, 360), uncage_end_raw=50, circle=None),
    dict(name="ATP4", condition="ATP", uncage_time="35 s",
         path=f"{ATP_35S}/AVG_Stack_ATP4.avi", angles=(0, 360), uncage_end_raw=50, circle=None),
    dict(name="ATP5", condition="ATP", uncage_time="35 s",
         path=f"{ATP_35S}/AVG_Stack_ATP5.avi", angles=(0, 360), uncage_end_raw=50, circle=None),
    dict(name="Suramin1", condition="suramin", uncage_time="full",
         path=f"{SURAMIN}/AVG_Stack_Suramin1.avi", angles=(0, 360), uncage_end_raw=400, circle=None),
    dict(name="suramin 2 (wound right)", condition="suramin", uncage_time="full",
         path=f"{SURAMIN}/AVG_Stack_ATP uncaging suramin 2.avi", angles=(90, 270), uncage_end_raw=400, circle=None),
    dict(name="suramin 3 (wound right)", condition="suramin", uncage_time="full",
         path=f"{SURAMIN}/AVG_Stack_ATP uncaging suramin 3.avi", angles=(90, 270), uncage_end_raw=400, circle=None),
]
for s in STACKS:
    s["group"] = f"{s['condition']}, uncage {s['uncage_time']}"
GROUP_COLORS = {"ATP, uncage full": "#2a78d6", "ATP, uncage 35 s": "#eb6834", "suramin, uncage full": "#1baf7a",
                "suramin, uncage 35 s": "#eda100"}

# ---- acquisition ----
PIXEL_SIZE_UM = 1.0          # 1 px = 1 um
FRAME_INTERVAL_S = 7.0       # one AVG frame = 10 raw frames x 0.7 s
RAW_FRAMES_PER_FRAME = 10

# ---- measurement ----
DETECT_FRAMES = (0, 1)       # circle is detected on the per-pixel minimum of these frames
RING_STEP = 25               # ring k covers r0 + k*RING_STEP +- RING_STEP/2 (px)
RING_MAX_OFFSET = 200
WEDGE_DEG = 11.25
MIN_PIXELS = 20              # a sector needs >= this many tissue pixels to be measured, and >= this many
                             # sheet pixels to be timed
FIT_MIN_OFFSET = 25          # the ring on the circle edge (offset 0) is never used for timing

# ---- sheet wave (shared with the wound notebook: ca_wave/sheet_wave.py, values used in the paper) ----
SHEET = dict(SHEET_SETTINGS)  # SMOOTH_SIGMA_PX, TISSUE_K, ACTIVE_RISE, ACTIVE_NOISE_K, RISE_WINDOW, OPEN_ITER, LATE_TOL,
                              # RECRUIT_WINDOW, ONSET_ONLY; the source is the lit circle while the light is on

# ---- wave decision ----
MIN_PEAK = 0.2               # a sector responds if its peak (I-min)/min >= this ...
NOISE_K = 4.0                # ... and >= NOISE_K x its own frame-to-frame noise,
                             # ... and its trace starts below half of its peak (it rises after the movie starts)
MIN_FRAC_RESPONDING = 0.5    # a ring is "reached" when >= this fraction of measured wedges respond
MIN_WAVE_RINGS = 3           # an outward wave must reach this many consecutive rings from 25 um (3 -> 75 um)
INITIAL_WINDOW_S = 21.0      # the immediate response = the rings reached within this time of the event (3 frames)
MIN_BEYOND_RINGS = 2         # an outward wave must reach this many rings beyond the immediate response (50 um)
N_PERMUTATIONS = 5000        # ring-label permutations for the outward-order test
ALPHA = 0.05
RANDOM_SEED = 0

# ---- plotting ----
NORM_VMAX = 1.0               # same colour scale for every heatmap of every stack
MONTAGE_FRAMES = [2, 3, 4, 5, 6, 8, 12, 20, 30]

OFFSETS = np.arange(0, RING_MAX_OFFSET + RING_STEP, RING_STEP)   # 0, 25, ..., 200
N_WEDGES = int(round(360 / WEDGE_DEG))
WEDGE_LO = np.arange(N_WEDGES) * WEDGE_DEG
WEDGE_LABELS = [f"{lo:g}-{lo + WEDGE_DEG:g}" for lo in WEDGE_LO]

OUT_ROOT.mkdir(parents=True, exist_ok=True)
for s in STACKS:   # if a folder was renamed, find the movie by its file name anywhere under DATA_ROOT
    if not (DATA_ROOT / s["path"]).exists():
        hits = sorted(DATA_ROOT.rglob(Path(s["path"]).name))
        if hits:
            s["path"] = str(hits[0].relative_to(DATA_ROOT))
            print(f"NOTE: {s['name']} is not at its listed path; using {s['path']}")
missing = [s["name"] for s in STACKS if not (DATA_ROOT / s["path"]).exists()]
if missing:
    print(f"WARNING: these stacks are not found under {DATA_ROOT} and will be skipped: {missing}")
STACKS_TO_RUN = [s for s in STACKS if s["name"] not in missing]
print(f"Normalization: {NORM_LABEL}")
print(f"Data root: {DATA_ROOT}")
print(f"Results:   {OUT_ROOT}")
print(f"Stacks:    {[s['name'] for s in STACKS_TO_RUN]}")

## Uncaging circle detection

Search the per-pixel minimum of frames 0 and 1 for a flat disk
that is brighter inside than outside along most of its edge, then refine it with a least-squares circle fit.
If a stack is detected wrongly, set `circle=(cx, cy, r)` for that row in `STACKS`.

In [ ]:
SEARCH_BOX = None                  # (x0, y0, x1, y1) px the circle's CENTER must lie in; None = whole image
RADIUS_RANGE = (40, 85)            # min/max circle radius to try (px)
EDGE_BAND_PX = 8
N_SECTORS = 16
SECTOR_QUANTILE = 25
MAX_INSIDE_BRIGHTNESS = 0.6


def fit_circle(points):
    # Algebraic least-squares circle fit (Kasa method) to boundary points.
    pts = np.asarray(points, dtype=float)
    x, y = pts[:, 0], pts[:, 1]
    A = np.column_stack([x, y, np.ones_like(x)])
    b = -(x**2 + y**2)
    D, E, F = np.linalg.lstsq(A, b, rcond=None)[0]
    cx, cy = -D / 2, -E / 2
    radius = np.sqrt(max(cx**2 + cy**2 - F, 0.0))
    residuals = np.hypot(x - cx, y - cy) - radius
    return cx, cy, radius, float(np.sqrt(np.mean(residuals**2)))


def _sector_ring_kernel(r_in, r_out, k, n_sectors):
    R = int(np.ceil(r_out))
    y, x = np.mgrid[-R:R + 1, -R:R + 1]
    d = np.hypot(x, y)
    sector = ((np.arctan2(y, x) + np.pi) / (2 * np.pi) * n_sectors).astype(int) % n_sectors
    m = ((d >= r_in) & (d < r_out) & (sector == k)).astype(float)
    return m[::-1, ::-1] / m.sum()


def _disk_kernel(r):
    R = int(np.ceil(r))
    y, x = np.mgrid[-R:R + 1, -R:R + 1]
    m = (np.hypot(x, y) < r).astype(float)
    return m / m.sum()


def detect_bright_circle(img, search_box=SEARCH_BOX, radius_range=RADIUS_RANGE, band=EDGE_BAND_PX,
                         n_sectors=N_SECTORS, quantile=SECTOR_QUANTILE,
                         max_inside_brightness=MAX_INSIDE_BRIGHTNESS, downsample=2):
    h, w = img.shape
    x0, y0, x1, y1 = search_box if search_box is not None else (0, 0, w, h)
    blur = gaussian(img.astype(float), sigma=2.0, preserve_range=True)
    bright_ref = np.percentile(blur, 99.5)

    small = downscale_local_mean(blur, (downsample, downsample))
    allowed = np.zeros_like(small, dtype=bool)
    allowed[y0 // downsample:y1 // downsample, x0 // downsample:x1 // downsample] = True
    best = None
    for r in np.arange(radius_range[0], radius_range[1] + 1, 2 * downsample):
        rs, bs = r / downsample, band / downsample
        contrast = np.percentile(np.stack([
            fftconvolve(small, _sector_ring_kernel(rs - bs, rs, k, n_sectors), "same")
            - fftconvolve(small, _sector_ring_kernel(rs, rs + bs, k, n_sectors), "same")
            for k in range(n_sectors)
        ]), quantile, axis=0)
        kd = _disk_kernel(0.8 * rs)
        mean_in = fftconvolve(small, kd, "same")
        std_in = np.sqrt(np.clip(fftconvolve(small**2, kd, "same") - mean_in**2, 0, None))
        score = contrast / (std_in + 2.0)
        m = int(np.ceil(rs + bs))
        valid = allowed & (contrast > 0) & (mean_in < max_inside_brightness * bright_ref)
        valid[:m] = valid[-m:] = False
        valid[:, :m] = valid[:, -m:] = False
        score = np.where(valid, score, -np.inf)
        iy, ix = np.unravel_index(np.argmax(score), score.shape)
        if np.isfinite(score[iy, ix]) and (best is None or score[iy, ix] > best[0]):
            best = (score[iy, ix], (ix + 0.5) * downsample, (iy + 0.5) * downsample, float(r),
                    contrast[iy, ix], std_in[iy, ix])
    if best is None:
        raise RuntimeError("No uncaging disk found; set circle=(cx, cy, r) for this stack in STACKS.")
    score, cx, cy, r, edge_contrast, std_in = best

    angles = np.linspace(0, 2 * np.pi, 180, endpoint=False)
    radial = np.arange(0.7 * r, 1.3 * r, 0.5)
    xs = cx + np.outer(np.cos(angles), radial)
    ys = cy + np.outer(np.sin(angles), radial)
    profiles = map_coordinates(blur, [ys, xs], order=1, mode="nearest")
    drops = -np.gradient(profiles, axis=1)
    idx = np.argmax(drops, axis=1)
    rows = np.arange(len(angles))
    strength = drops[rows, idx]
    keep = strength >= np.percentile(strength, 25)
    edge_pts = np.column_stack([ys[rows, idx], xs[rows, idx]])[keep]
    cy_fit, cx_fit, radius, rmse = fit_circle(edge_pts)
    return {"cx": cx_fit, "cy": cy_fit, "radius": radius, "fit_rmse_px": rmse,
            "edge_contrast": float(edge_contrast), "score": float(score)}

## Measurement and wave test

For every stack:

**Sheet pixels** (`ca_wave/sheet_wave.py`, the same code and settings for the wound and the uncaging movies).
The movies show two kinds of Ca²⁺ signal: the wave, one connected sheet of responding cells that grows out from
the source (here the lit circle, while the light is on), and single cells that flash on their own before, during and long after the wave (they flash just as
much under suramin, where there is no wave, and before the wound is made). A sector average mixes the two, so a
flash can set a sector's arrival. Each sector is therefore measured on its sheet pixels only:
1. every frame is lightly smoothed (`SMOOTH_SIGMA_PX` = 2 px) and each pixel's trace is normalized as
   `(I - min)/min`; tissue pixels are those whose minimum over time is clearly above the dark background
   (`TISSUE_K`);
2. a pixel is **active** in a frame when its `(I - min)/min` has risen by at least 0.3 (`ACTIVE_RISE`) and 4 × its
   own frame-to-frame noise (`ACTIVE_NOISE_K`) above its lowest value of the last 3 frames (`RISE_WINDOW`); a
   cell that is bright and slowly dimming is not active;
3. the **sheet** starts at the source and grows: in every frame, each region of pixels that *start* to be active
   in that frame (`ONSET_ONLY`) joins the sheet if it touches the front, i.e. the source or pixels the sheet
   reached in the last 3 frames (`RECRUIT_WINDOW`). Pixels reached long ago do not recruit, so cells that flash
   next to the sheet minutes after it passed cannot chain a false wave outward;
4. a pixel that joined the sheet is **timed** unless it had already been active for more than one frame
   (`LATE_TOL`) before the sheet reached it (its own earlier flash). Active pixels the sheet never reaches are
   flashes and are left out.

Figure 5 shows, for every movie, the sheet pixels, when the sheet reached them, and the pixels left out.

Then, on the sheet pixels:

1. **Sectors.** The area around the circle is cut into 11.25° wedges × 25-px rings (offset 0 to 200 px from the
   circle edge). Each sector's value in a frame is the mean intensity of its sheet pixels (the ring on the circle edge,
   offset 0, which is never used for timing, is measured on all its pixels). A sector is measured
   if it has ≥ `MIN_PIXELS` tissue pixels, and timed if it has ≥ `MIN_PIXELS` sheet pixels (otherwise it counts
   as not responding). Steps 3-6 use the measured wedges (`angles`).
2. **Normalization:** `(I - min)/min`, with the minimum taken over all frames of that sector.
3. **Response.** A sector responds if its normalized peak ≥ `MIN_PEAK` and ≥ `NOISE_K` × its frame-to-frame
   noise (robust SD of frame differences), **and** its trace starts below half of its peak (`starts_low`), i.e. it
   rises after the movie starts. Arrival = the frame where the trace first crosses half of its peak on the way up,
   interpolated between frames.
4. **Reach.** A ring is reached when ≥ `MIN_FRAC_RESPONDING` of the measured wedges respond. Reach = the last ring
   of the unbroken run that starts at 25 px. The **immediate response** is the same run counted only over
   arrivals within `INITIAL_WINDOW_S` (21 s) of light-on.
5. **Outward-order test.** Kendall τ between distance and arrival over every responding sector of the measured
   wedges (25-200 px). The null distribution comes from shuffling which distance each ring sits at, using the same
   shuffle for all wedges. An outward wave gives τ > 0 with a small p. Activity coming in from the image edge gives
   τ < 0.
6. **Outward wave** = reach ≥ `MIN_WAVE_RINGS` rings, reach ≥ the immediate response + `MIN_BEYOND_RINGS` rings,
   **and** τ > 0 **and** p < `ALPHA`. Only then is a speed reported: Theil-Sen fit of the median arrival time per
   ring vs distance over the reached rings, speed = 1 / slope (µm/s), with its 95 % interval.


In [ ]:
def load_avi(path):
    cap = cv2.VideoCapture(str(path))
    frames = []
    while True:
        ok, img = cap.read()
        if not ok:
            break
        frames.append(img[..., 0])          # grayscale stored as 3 identical channels
    cap.release()
    if not frames:
        raise IOError(f"Could not read any frames from {path}")
    return np.stack(frames).astype(float)


def sector_means(stack, cx, cy, r0, pix=None, tissue=None):
    # Mean intensity of every (wedge, ring) sector in every frame, over the pixels in `pix` (the sheet pixels;
    # all pixels if None) -> F[wedge, ring, frame], n_px[wedge, ring] = pixels averaged,
    # n_tissue[wedge, ring] = tissue pixels in the sector (all pixels if `tissue` is None).
    n_frames, h, w = stack.shape
    yy, xx = np.mgrid[:h, :w]
    dist_from_edge = np.hypot(xx - cx, yy - cy) - r0
    angle = np.degrees(np.arctan2(yy - cy, xx - cx)) % 360
    wedge_idx = np.floor(angle / WEDGE_DEG).astype(int) % N_WEDGES
    ring_idx = np.floor((dist_from_edge + RING_STEP / 2) / RING_STEP).astype(int)
    in_rings = (ring_idx >= 0) & (ring_idx < len(OFFSETS))
    n_bins = N_WEDGES * len(OFFSETS)
    lab_all = wedge_idx * len(OFFSETS) + ring_idx
    used = in_rings & ((pix if pix is not None else True) | (ring_idx == 0))   # the edge ring: all pixels
    label = lab_all[used]
    n_px = np.bincount(label, minlength=n_bins)
    sums = np.stack([np.bincount(label, weights=f[used], minlength=n_bins) for f in stack])
    F = (sums / np.maximum(n_px, 1)).reshape(n_frames, N_WEDGES, len(OFFSETS)).transpose(1, 2, 0)
    n_px = n_px.reshape(N_WEDGES, len(OFFSETS))
    t_used = in_rings & ((tissue if tissue is not None else True) | (ring_idx == 0))
    n_tissue = np.bincount(lab_all[t_used], minlength=n_bins).reshape(N_WEDGES, len(OFFSETS))
    F[n_px < MIN_PIXELS] = np.nan
    return F, n_px, n_tissue


def frame_noise(norm):
    # Robust SD of frame-to-frame changes (per sector).
    d = np.diff(norm, axis=-1)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        mad = np.nanmedian(np.abs(d - np.nanmedian(d, axis=-1, keepdims=True)), axis=-1)
    return 1.4826 * mad / np.sqrt(2)


def half_max_arrival(norm):
    # Peak and half-max arrival frame (linear interpolation between frames) along the last axis.
    filled = np.nan_to_num(norm, nan=-np.inf)
    peak = filled.max(axis=-1)
    thr = 0.5 * peak
    i = np.argmax(filled >= thr[..., None], axis=-1)
    v1 = np.take_along_axis(filled, i[..., None], -1)[..., 0]
    v0 = np.take_along_axis(filled, np.maximum(i - 1, 0)[..., None], -1)[..., 0]
    rising = (i > 0) & (v1 > v0)
    with np.errstate(invalid="ignore"):     # sectors outside the image are all -inf
        frac = np.divide(thr - v0, v1 - v0, out=np.zeros_like(thr), where=rising)
    arrival = np.where(i > 0, i - 1 + frac, 0.0)
    return np.where(np.isfinite(peak), peak, np.nan), arrival


def wedges_in_range(lo, hi):
    hi_u = hi + 360 if hi <= lo else hi
    centre = WEDGE_LO + WEDGE_DEG / 2
    centre = np.where(centre < lo, centre + 360, centre)
    return (centre > lo) & (centre < hi_u)


def leading_run(mask):
    # Length of the unbroken run of True values at the start of mask.
    return int(np.argmin(np.append(mask, False)))


def normalize(F):
    # (I - min)/min per sector for F[wedge, ring, frame]; returns (normalized trace, min over time).
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        fmin = np.nanmin(F, axis=-1)
    return (F - fmin[..., None]) / fmin[..., None], fmin


def analyze_stack(cfg, rng, stack=None):
    if stack is None:
        stack = load_avi(DATA_ROOT / cfg["path"])
    if cfg.get("circle"):
        cx, cy, r0 = cfg["circle"]
        circle = {"cx": cx, "cy": cy, "radius": r0, "fit_rmse_px": np.nan, "edge_contrast": np.nan, "source": "manual"}
    else:
        circle = detect_bright_circle(stack[list(DETECT_FRAMES)].min(axis=0))
        circle["source"] = "auto"
    cx, cy, r0 = circle["cx"], circle["cy"], circle["radius"]

    # the sheet wave: pixels reached by the sheet that grows from the lit circle (ca_wave/sheet_wave.py)
    yy, xx = np.mgrid[:stack.shape[1], :stack.shape[2]]
    lit = np.hypot(xx - cx, yy - cy) <= r0
    light_frames = cfg["uncage_end_raw"] / RAW_FRAMES_PER_FRAME          # the lit circle is the source while lit
    sheet = sheet_pixels(stack, seed=lambda t: lit if t < light_frames else None, settings=SHEET)
    F, n_px, n_tissue = sector_means(stack, cx, cy, r0, pix=sheet["timed"], tissue=sheet["tissue"])
    valid = n_tissue >= MIN_PIXELS           # sectors with enough tissue are measured (timed or not)
    norm, Fmin = normalize(F)               # (I - min)/min per sector
    noise = frame_noise(norm)
    peak, arrival = half_max_arrival(norm)
    threshold = np.maximum(MIN_PEAK, NOISE_K * noise)
    with np.errstate(invalid="ignore"):
        starts_low = norm[..., 0] < 0.5 * peak   # must rise after the start; a trace that starts high and dims is not a response
    responds = np.isfinite(peak) & (peak >= threshold) & starts_low

    # measured wedges, rings >= FIT_MIN_OFFSET
    in_range = wedges_in_range(*cfg["angles"])
    fit_rings = OFFSETS >= FIT_MIN_OFFSET
    offs = OFFSETS[fit_rings]
    resp_m = responds[in_range][:, fit_rings]
    valid_m = valid[in_range][:, fit_rings]
    arr_m = arrival[in_range][:, fit_rings]

    frac_resp = resp_m.sum(0) / np.maximum(valid_m.sum(0), 1)
    n_reached = leading_run(frac_resp >= MIN_FRAC_RESPONDING)
    reach_um = float(offs[n_reached - 1] * PIXEL_SIZE_UM) if n_reached else 0.0
    median_arrival = np.array([np.median(arr_m[resp_m[:, k], k]) if resp_m[:, k].any() else np.nan
                               for k in range(len(offs))])

    # outward-order test: Kendall tau(distance, arrival) vs ring-label permutations
    w_idx, r_idx = np.nonzero(resp_m)
    y = arr_m[w_idx, r_idx]
    tau, p, null = np.nan, np.nan, np.array([])
    if len(np.unique(r_idx)) >= 3 and np.ptp(y) > 0:
        tau = kendalltau(offs[r_idx], y).statistic
        null = np.array([kendalltau(offs[rng.permutation(len(offs))][r_idx], y).statistic
                         for _ in range(N_PERMUTATIONS)])
        p = (1 + np.sum(null >= tau)) / (1 + N_PERMUTATIONS)

    # the immediate response: rings where >= MIN_FRAC_RESPONDING of the wedges respond within INITIAL_WINDOW_S
    early_m = resp_m & (arr_m <= INITIAL_WINDOW_S / FRAME_INTERVAL_S)
    n_initial = leading_run(early_m.sum(0) / np.maximum(valid_m.sum(0), 1) >= MIN_FRAC_RESPONDING)
    initial_reach_um = float(offs[n_initial - 1] * PIXEL_SIZE_UM) if n_initial else 0.0

    is_wave = bool(n_reached >= MIN_WAVE_RINGS and n_reached >= n_initial + MIN_BEYOND_RINGS
                   and np.isfinite(tau) and tau > 0 and p < ALPHA)
    if is_wave:
        verdict = "outward wave"
    elif n_reached < MIN_WAVE_RINGS:
        verdict = f"no outward wave: responses reach only {reach_um:.0f} um"
    elif n_reached < n_initial + MIN_BEYOND_RINGS:
        verdict = (f"no outward wave: the response does not spread beyond the immediate response "
                   f"({initial_reach_um:.0f} um within {INITIAL_WINDOW_S:.0f} s; reach {reach_um:.0f} um)")
    else:
        verdict = f"no outward wave: arrival not later with distance (tau={tau:.2f}, p={p:.3f})"

    speed = speed_lo = speed_hi = np.nan
    fit = None
    if is_wave:
        x_um = offs[:n_reached] * PIXEL_SIZE_UM
        t_s = median_arrival[:n_reached] * FRAME_INTERVAL_S
        slope, intercept, lo_slope, hi_slope = theilslopes(t_s, x_um)   # s per um
        fit = (slope, intercept, x_um)
        if slope > 0:
            speed = 1.0 / slope
            speed_lo = 1.0 / hi_slope if hi_slope > 0 else np.nan
            speed_hi = 1.0 / lo_slope if lo_slope > 0 else np.inf
        else:
            is_wave, verdict = False, "no outward wave: median arrival does not increase with distance"

    # descriptive per-wedge numbers (not used for the decision)
    wedge_rows = []
    for j, w in enumerate(np.nonzero(in_range)[0]):
        r = resp_m[j]
        nc = leading_run(r)
        tau_w = kendalltau(offs[r], arr_m[j, r]).statistic if r.sum() >= 3 and np.ptp(arr_m[j, r]) > 0 else np.nan
        speed_w = np.nan
        if nc >= MIN_WAVE_RINGS:
            s_w = theilslopes(arr_m[j, :nc] * FRAME_INTERVAL_S, offs[:nc] * PIXEL_SIZE_UM)[0]
            speed_w = 1.0 / s_w if s_w > 0 else np.nan
        wedge_rows.append({"sub_wedge": WEDGE_LABELS[w], "n_responding_rings": int(r.sum()),
                           "consecutive_rings_from_25px": nc,
                           "reach_um": float(offs[nc - 1] * PIXEL_SIZE_UM) if nc else 0.0,
                           "kendall_tau": tau_w, "speed_um_per_s": speed_w})

    return dict(cfg=cfg, stack=stack, circle=circle, sheet=sheet, n_tissue=n_tissue, valid=valid,
                F=F, Fmin=Fmin, n_px=n_px, norm=norm,
                noise=noise, peak=peak, starts_low=starts_low,
                arrival=arrival, threshold=threshold, responds=responds, in_range=in_range, offs=offs,
                resp_m=resp_m, arr_m=arr_m, frac_resp=frac_resp, n_reached=n_reached, reach_um=reach_um,
                n_initial=n_initial, initial_reach_um=initial_reach_um, median_arrival=median_arrival, tau=tau, p=p, null=null, is_wave=is_wave, verdict=verdict,
                speed=speed, speed_lo=speed_lo, speed_hi=speed_hi, fit=fit, wedge_df=pd.DataFrame(wedge_rows))

### Saving and plotting helpers

In [ ]:
def save_figure(fig, folder, name, dpi=200):
    folder.mkdir(parents=True, exist_ok=True)
    for ext in ("png", "pdf"):
        fig.savefig(folder / f"{name}.{ext}", dpi=dpi, bbox_inches="tight")


def save_tables(res, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    n_frames = res["norm"].shape[-1]
    c = res["circle"]
    pd.DataFrame([{k: c[k] for k in ("cx", "cy", "radius", "fit_rmse_px", "edge_contrast", "source")}]).to_csv(
        out_dir / "circle.csv", index=False)

    w, r, f = np.meshgrid(np.arange(N_WEDGES), np.arange(len(OFFSETS)), np.arange(n_frames), indexing="ij")
    pd.DataFrame({
        "sub_wedge": np.array(WEDGE_LABELS)[w.ravel()], "in_measured_range": res["in_range"][w.ravel()],
        "offset_px": OFFSETS[r.ravel()], "frame": f.ravel(), "time_s": f.ravel() * FRAME_INTERVAL_S,
        "mean_intensity": res["F"].ravel(),
        "min_over_time": res["Fmin"][w.ravel(), r.ravel()],
        "normalized_intensity": res["norm"].ravel(),
    }).to_csv(out_dir / "sector_intensity_over_time.csv", index=False)

    w2, r2 = np.meshgrid(np.arange(N_WEDGES), np.arange(len(OFFSETS)), indexing="ij")
    pd.DataFrame({
        "sub_wedge": np.array(WEDGE_LABELS)[w2.ravel()], "in_measured_range": res["in_range"][w2.ravel()],
        "offset_px": OFFSETS[r2.ravel()], "n_tissue_pixels": res["n_tissue"].ravel(),
        "n_sheet_pixels": res["n_px"].ravel(), "measured": res["valid"].ravel(), "peak_normalized": res["peak"].ravel(),
        "noise_normalized": res["noise"].ravel(), "response_threshold": res["threshold"].ravel(),
        "starts_below_half_peak": res["starts_low"].ravel(),
        "responds": res["responds"].ravel(), "arrival_frame": np.where(res["responds"], res["arrival"], np.nan).ravel(),
        "arrival_s": np.where(res["responds"], res["arrival"] * FRAME_INTERVAL_S, np.nan).ravel(),
    }).to_csv(out_dir / "sector_arrival_times.csv", index=False)

    pd.DataFrame({
        "offset_px": res["offs"], "offset_um": res["offs"] * PIXEL_SIZE_UM,
        "fraction_wedges_responding": res["frac_resp"],
        "fraction_tissue_in_sheet": sheet_fraction(res),
        "reached": np.arange(len(res["offs"])) < res["n_reached"],
        "median_arrival_frame": res["median_arrival"], "median_arrival_s": res["median_arrival"] * FRAME_INTERVAL_S,
    }).to_csv(out_dir / "ring_summary.csv", index=False)
    res["wedge_df"].to_csv(out_dir / "per_wedge_summary.csv", index=False)


def sheet_fraction(res):
    # per ring (>= FIT_MIN_OFFSET, measured wedges): sheet pixels / tissue pixels
    fit_rings = OFFSETS >= FIT_MIN_OFFSET
    n_s = res["n_px"][res["in_range"]][:, fit_rings].sum(0)
    n_t = res["n_tissue"][res["in_range"]][:, fit_rings].sum(0)
    return n_s / np.maximum(n_t, 1)


def plot_sheet_maps(res):
    # which pixels the sheet reached and when, and which were left out
    cfg, c, sh = res["cfg"], res["circle"], res["sheet"]
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    t_s = np.where(sh["timed"], sh["t_reach"] * FRAME_INTERVAL_S, np.nan)
    im = axes[0].imshow(t_s, cmap="viridis")
    fig.colorbar(im, ax=axes[0], shrink=0.8, label="time the sheet reached the pixel (s)")
    axes[0].set_title("sheet pixels (timed): when the sheet reached them", fontsize=10)
    cls = np.zeros(sh["tissue"].shape)
    cls[sh["tissue"]] = 1
    cls[sh["timed"]] = 2
    cls[sh["early"]] = 3
    cls[sh["flash_only"]] = 4
    colors = ["black", "0.35", "tab:green", "tab:orange", "tab:red"]
    axes[1].imshow(cls, cmap=ListedColormap(colors), vmin=-0.5, vmax=4.5, interpolation="nearest")
    labels = ["not tissue", "tissue, not reached", "sheet (timed)", "active before the sheet came (not timed)",
              "active, never reached by the sheet (flashes)"]
    axes[1].legend(handles=[Patch(color=cc, label=l) for cc, l in zip(colors, labels)], fontsize=7, loc="lower left")
    axes[1].set_title("pixel classes", fontsize=10)
    for ax in axes:
        draw_measured_half(ax, c["cx"], c["cy"], c["radius"], cfg["angles"])
        ax.axis("off")
    fig.suptitle(f"{cfg['name']}: sheet wave (only green pixels are used for the sector traces)", y=1.0)
    fig.tight_layout()
    return fig


def draw_measured_half(ax, cx, cy, r0, angles, scale=1.0, color="orange"):
    lo, hi = angles
    hi_u = hi + 360 if hi <= lo else hi
    outer = r0 + RING_MAX_OFFSET + RING_STEP / 2
    ax.add_patch(Wedge((cx * scale, cy * scale), outer * scale, lo, hi_u, width=(outer - r0) * scale,
                       facecolor="none", edgecolor=color, linewidth=1.2, linestyle="--"))
    ax.add_patch(Circle((cx * scale, cy * scale), r0 * scale, fill=False, edgecolor="cyan", linewidth=1.2))


def plot_overview(res):
    # Frame 1 with circle + measured wedges, then normalized frames on a fixed scale.
    cfg, st, c = res["cfg"], res["stack"], res["circle"]
    n_frames = st.shape[0]
    frames = [f for f in MONTAGE_FRAMES if f < n_frames]
    # per-pixel minimum over time, after light smoothing so single-pixel noise does not set the minimum
    base = gaussian(st, sigma=(0, 1.5, 1.5), preserve_range=True).min(axis=0)
    fig, axes = plt.subplots(2, 5, figsize=(20, 8.4))
    axes = axes.ravel()
    vmin, vmax = np.percentile(st[1], [1, 99.5])
    axes[0].imshow(st[1], cmap="gray", vmin=vmin, vmax=vmax)
    draw_measured_half(axes[0], c["cx"], c["cy"], c["radius"], cfg["angles"])
    axes[0].set_title(f"frame 1 + circle ({c['source']}) + measured wedges", fontsize=9)
    for ax, f in zip(axes[1:], frames):
        ax.imshow((st[f] - base) / np.maximum(base, 5), cmap="magma", vmin=0, vmax=NORM_VMAX)
        draw_measured_half(ax, c["cx"], c["cy"], c["radius"], cfg["angles"])
        ax.set_title(f"{NORM_LABEL} frame {f} ({f * FRAME_INTERVAL_S:.0f} s)", fontsize=9)
    for ax in axes:
        ax.axis("off")
    for ax in axes[1 + len(frames):]:
        ax.set_visible(False)
    fig.suptitle(f"{cfg['name']} ({cfg['group']}): {NORM_LABEL}, fixed scale 0-{NORM_VMAX:g}; "
                 f"dashed = measured wedges", y=1.0)
    fig.tight_layout()
    return fig


def plot_angle_maps(res):
    # Full 360 deg (display only): peak and arrival time per sector, to check the measured wedges.
    cfg = res["cfg"]
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.2))
    ext = [0, 360, OFFSETS[0] - RING_STEP / 2, OFFSETS[-1] + RING_STEP / 2]
    im0 = axes[0].imshow(res["peak"].T, aspect="auto", origin="lower", extent=ext, cmap="magma", vmin=0, vmax=NORM_VMAX)
    fig.colorbar(im0, ax=axes[0], label=f"peak {NORM_LABEL}")
    arr_s = np.where(res["responds"], res["arrival"] * FRAME_INTERVAL_S, np.nan)
    im1 = axes[1].imshow(arr_s.T, aspect="auto", origin="lower", extent=ext, cmap="viridis")
    fig.colorbar(im1, ax=axes[1], label="arrival time (s), responding sectors only")
    lo, hi = cfg["angles"]
    spans = [(lo, hi)] if hi > lo else [(lo, 360), (0, hi)]
    for ax in axes:
        for a, b in spans:
            ax.axvspan(a, b, facecolor="none", edgecolor="orange", lw=2.5, ls="--")
        ax.axhline(FIT_MIN_OFFSET - RING_STEP / 2, color="white", lw=0.8, ls=":")
        ax.set_xlabel("angle (deg, 0 = 3 o'clock, clockwise)")
        ax.set_ylabel("distance from circle edge (px)")
        ax.set_xticks(np.arange(0, 361, 45))
    axes[0].set_title(f"peak {NORM_LABEL}, full circle (orange = measured wedges)")
    axes[1].set_title("arrival time: an outward wave is earliest at small distance")
    fig.suptitle(cfg["name"], y=1.02)
    fig.tight_layout()
    return fig


def plot_kymographs(res):
    # normalized intensity (distance x frame) per measured wedge, fixed colour scale, arrival points.
    cfg = res["cfg"]
    idx = np.nonzero(res["in_range"])[0]
    n_frames = res["norm"].shape[-1]
    ncols = 4
    nrows = int(np.ceil(len(idx) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.4 * ncols, 3.3 * nrows), squeeze=False, layout="constrained")
    axes = axes.ravel()
    uncage_end = cfg["uncage_end_raw"] / RAW_FRAMES_PER_FRAME
    for ax, w in zip(axes, idx):
        im = ax.imshow(res["norm"][w].T, aspect="auto", origin="lower", cmap="magma", vmin=0, vmax=NORM_VMAX,
                       extent=[OFFSETS[0] - RING_STEP / 2, OFFSETS[-1] + RING_STEP / 2, -0.5, n_frames - 0.5])
        rsp = res["responds"][w] & (OFFSETS >= FIT_MIN_OFFSET)
        ax.plot(OFFSETS[rsp], res["arrival"][w][rsp], "o", color="white", mec="black", ms=5)
        if uncage_end < n_frames:
            ax.axhline(uncage_end, color="cyan", ls="--", lw=1)
        ax.axvline(FIT_MIN_OFFSET - RING_STEP / 2, color="white", lw=0.6, ls=":")
        ax.set_title(f"{WEDGE_LABELS[w]} deg  ({rsp.sum()} rings respond)", fontsize=9)
        ax.set_xlabel("offset (px)")
        ax.set_ylabel("frame")
    for ax in axes[len(idx):]:
        ax.axis("off")
    fig.colorbar(im, ax=axes[:len(idx)].tolist(), shrink=0.6, label=f"{NORM_LABEL} (same scale for all stacks)")
    fig.suptitle(f"{cfg['name']}: {NORM_LABEL} per wedge; white dots = half-max arrival of responding sectors; "
                 f"cyan dashed = uncaging end")
    return fig


def plot_arrival_vs_distance(res):
    cfg = res["cfg"]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw={"width_ratios": [2, 1]})
    ax = axes[0]
    offs_um = res["offs"] * PIXEL_SIZE_UM
    w_idx, r_idx = np.nonzero(res["resp_m"])
    jitter = np.random.default_rng(1).uniform(-3, 3, len(r_idx))
    ax.plot(offs_um[r_idx] + jitter, res["arr_m"][w_idx, r_idx] * FRAME_INTERVAL_S, "o", color="0.6", ms=4,
            alpha=0.6, label="responding sector (one per wedge)")
    ax.plot(offs_um, res["median_arrival"] * FRAME_INTERVAL_S, "s-", color="black", ms=7, label="median per ring")
    if res["n_reached"]:
        ax.axvspan(offs_um[0] - 12.5, offs_um[res["n_reached"] - 1] + 12.5, color="gold", alpha=0.15,
                   label=f"reached rings (>= {MIN_FRAC_RESPONDING:.0%} of wedges respond)")
    if res["is_wave"]:
        slope, intercept, x_um = res["fit"]
        ax.plot(x_um, slope * x_um + intercept, "-", color="tab:red", lw=2.5,
                label=f"Theil-Sen: {res['speed']:.2f} um/s (95% CI {res['speed_lo']:.2f}-{res['speed_hi']:.2f})")
    ax.set_xlabel("distance from circle edge (um)")
    ax.set_ylabel("arrival time (s)")
    ax.set_xlim(0, OFFSETS[-1] * PIXEL_SIZE_UM + 15)
    ax.legend(fontsize=8, loc="upper left")
    ax.set_title(f"{cfg['name']}: {res['verdict']}", fontsize=10)

    ax = axes[1]
    if res["null"].size:
        ax.hist(res["null"], bins=40, color="0.75")
        ax.axvline(res["tau"], color="tab:red", lw=2, label=f"observed tau = {res['tau']:.2f}")
        ax.set_title(f"outward-order test: p = {res['p']:.4f}", fontsize=10)
        ax.legend(fontsize=8)
    else:
        ax.text(0.5, 0.5, "too few responding rings\nfor the order test", ha="center", va="center",
                transform=ax.transAxes)
    ax.set_xlabel("Kendall tau (distance vs arrival)")
    ax.set_ylabel("ring-label permutations")
    fig.tight_layout()
    return fig

## Run every stack

Each stack gets its own folder in `OUT_ROOT` with:

- `circle.csv`
- `sector_intensity_over_time.csv` (every wedge × ring × frame, full circle: raw mean, min over time, normalized; `in_measured_range` marks the measured wedges; the means are over sheet pixels)
- `sector_arrival_times.csv` (tissue and sheet pixel counts, measured, peak, noise, threshold, starts below half
  peak, responds, arrival)
- `ring_summary.csv` (also the fraction of tissue pixels in the sheet per ring)
- `per_wedge_summary.csv`
- figures in `figures/` (5: the sheet pixels)

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
results = {}
for cfg in STACKS_TO_RUN:
    res = analyze_stack(cfg, rng)
    results[cfg["name"]] = res
    out_dir = OUT_ROOT / cfg["name"]
    save_tables(res, out_dir)
    c = res["circle"]
    print(f"=== {cfg['name']} ({cfg['group']}): {res['norm'].shape[-1]} frames, circle ({c['source']}) "
          f"centre=({c['cx']:.1f}, {c['cy']:.1f}) r={c['radius']:.1f}px, measured angles {cfg['angles']}")
    print(f"    fraction of wedges responding per ring (25..200 px): {np.round(res['frac_resp'], 2).tolist()}")
    print(f"    immediate response {res['initial_reach_um']:.0f} um (within {INITIAL_WINDOW_S:.0f} s), reach = {res['reach_um']:.0f} um, tau = {res['tau']:.2f}, p = {res['p']:.4f}")
    print(f"    -> {res['verdict']}" + (f", speed = {res['speed']:.2f} um/s "
                                        f"(95% CI {res['speed_lo']:.2f}-{res['speed_hi']:.2f})" if res["is_wave"] else ""))

    for fig_fn, fig_name in [(plot_overview, "1_overview_normalized_frames"), (plot_angle_maps, "2_full_circle_maps"),
                             (plot_kymographs, "3_kymographs_per_wedge"),
                             (plot_arrival_vs_distance, "4_arrival_vs_distance"), (plot_sheet_maps, "5_sheet_pixels")]:
        fig = fig_fn(res)
        save_figure(fig, out_dir / "figures", fig_name)
        plt.show()

## Summary of all stacks

In [ ]:
summary_df = pd.DataFrame([{
    "dataset": name, "condition": res["cfg"]["condition"], "uncage_time": res["cfg"]["uncage_time"],
    "group": res["cfg"]["group"], "uncage_end_raw_frame": res["cfg"]["uncage_end_raw"], "n_frames": res["norm"].shape[-1],
    "circle_cx": res["circle"]["cx"], "circle_cy": res["circle"]["cy"], "circle_r_px": res["circle"]["radius"],
    "circle_source": res["circle"]["source"], "measured_angles": f"{res['cfg']['angles'][0]}-{res['cfg']['angles'][1]}",
    "initial_response_reach_um": res["initial_reach_um"], "response_reach_um": res["reach_um"], "kendall_tau": res["tau"], "order_test_p": res["p"],
    "outward_wave": res["is_wave"], "verdict": res["verdict"],
    "wave_reach_um": res["reach_um"] if res["is_wave"] else 0.0,
    "speed_um_per_s": res["speed"], "speed_ci_low": res["speed_lo"], "speed_ci_high": res["speed_hi"],
    "median_per_wedge_speed_um_per_s": res["wedge_df"]["speed_um_per_s"].median() if res["is_wave"] else np.nan,
} for name, res in results.items()])
summary_df.to_csv(OUT_ROOT / "wave_summary_all_stacks.csv", index=False)
print(f"Saved: {OUT_ROOT / 'wave_summary_all_stacks.csv'}")
summary_df[["dataset", "group", "outward_wave", "verdict", "response_reach_um", "kendall_tau", "order_test_p",
            "speed_um_per_s", "speed_ci_low", "speed_ci_high"]].round(3)

## Pooled by group (condition × uncage time)

Arrival time vs distance for every stack with an outward wave, each aligned so its first ring (25 µm) is at 0 s.
The group mean ± SD is drawn where every wave in that group reached the ring. Stacks without an outward
wave contribute **wave reach = 0** and no speed.

In [ ]:
groups = list(dict.fromkeys(s["group"] for s in STACKS_TO_RUN))
offs_um = OFFSETS[OFFSETS >= FIT_MIN_OFFSET] * PIXEL_SIZE_UM

pooled_rows = []
fig, axes = plt.subplots(1, 3, figsize=(21, 5.5), gridspec_kw={"width_ratios": [2.2, 1, 1]})
ax = axes[0]
for cond in groups:
    color = GROUP_COLORS.get(cond, "0.3")
    waves = [r for r in results.values() if r["cfg"]["group"] == cond and r["is_wave"]]
    aligned = []
    for r in waves:
        t = (r["median_arrival"] - r["median_arrival"][0]) * FRAME_INTERVAL_S
        t[r["n_reached"]:] = np.nan
        aligned.append(t)
        ax.plot(offs_um, t, "o-", color=color, alpha=0.45, lw=1.2, ms=4)
    if aligned:
        A = np.vstack(aligned)
        complete = np.all(np.isfinite(A), axis=0)
        mean_t, sd_t = np.nanmean(A, axis=0), np.nanstd(A, axis=0)
        ax.plot(offs_um[complete], mean_t[complete], "-", color=color, lw=3,
                label=f"{cond}: mean of {len(waves)} wave(s)")
        ax.fill_between(offs_um[complete], (mean_t - sd_t)[complete], (mean_t + sd_t)[complete], color=color, alpha=0.15)
        for x, m, s, n in zip(offs_um, mean_t, sd_t, np.isfinite(A).sum(0)):
            pooled_rows.append({"group": cond, "offset_um": x, "mean_aligned_arrival_s": m,
                                "sd_aligned_arrival_s": s, "n_waves_reaching": int(n)})
    no_wave = [r["cfg"]["name"] for r in results.values() if r["cfg"]["group"] == cond and not r["is_wave"]]
    if no_wave:
        ax.plot([], [], " ", label=f"{cond}: no outward wave in {', '.join(no_wave)}")
ax.set_xlabel("distance from circle edge (um)")
ax.set_ylabel("arrival time from first ring (s)")
ax.set_title("Arrival time vs distance (outward waves only)")
ax.legend(fontsize=8, loc="upper left")

for ax, col, ylabel in [(axes[1], "wave_reach_um", "outward wave reach (um)"),
                        (axes[2], "speed_um_per_s", "wave speed (um/s)")]:
    for i, cond in enumerate(groups):
        sub = summary_df[summary_df["group"] == cond]
        waves, no_waves = sub[sub["outward_wave"]], sub[~sub["outward_wave"]]
        x = np.full(len(sub), i) + np.linspace(-0.12, 0.12, len(sub))
        y = sub[col].fillna(0) if col == "wave_reach_um" else sub[col]
        ax.plot(x, y, "o", color=GROUP_COLORS.get(cond, "0.3"), ms=9, mec="black")
        for k, (xi, yi, name, wave) in enumerate(zip(x, y, sub["dataset"], sub["outward_wave"])):
            if wave:
                ax.annotate(name, (xi, yi), fontsize=7, xytext=(6, 2 - 10 * (k % 2)), textcoords="offset points")
        if len(no_waves):
            ax.text(i, 0.5, "no outward wave:\n" + "\n".join(no_waves["dataset"]), fontsize=7, color="0.35",
                    ha="center", va="center", transform=ax.get_xaxis_transform())
    ax.set_xticks(range(len(groups)))
    ax.set_xticklabels([g.replace(", ", "\n") for g in groups])
    ax.set_xlim(-0.5, len(groups) - 0.5)
    ax.set_ylim(bottom=0)
    ax.set_ylabel(ylabel)
axes[1].set_title("Wave reach (0 = no outward wave)")
axes[2].set_title("Speed (outward waves only)")
fig.tight_layout()
save_figure(fig, OUT_ROOT / "figures", "pooled_by_group")
plt.show()

pooled_df = pd.DataFrame(pooled_rows)
pooled_df.to_csv(OUT_ROOT / "pooled_arrival_by_group.csv", index=False)

group_df = summary_df.groupby("group").agg(
    n_stacks=("dataset", "size"), n_outward_waves=("outward_wave", "sum"),
    mean_wave_reach_um=("wave_reach_um", "mean"), mean_speed_um_per_s=("speed_um_per_s", "mean"),
    sd_speed_um_per_s=("speed_um_per_s", "std"),
).reindex(groups)
group_df.to_csv(OUT_ROOT / "summary_by_group.csv")
print(f"Saved: {OUT_ROOT / 'pooled_arrival_by_group.csv'}")
print(f"Saved: {OUT_ROOT / 'summary_by_group.csv'}")
group_df.round(3)

## All stacks in one figure

**Top row.** Median arrival time per ring vs distance, one panel per group, all on the same time axis.
- Filled markers with a solid line show an outward wave; the line is the Theil-Sen fit.
- Hollow markers with a dashed line mean no outward wave.

**Bottom row.** Every stack side by side.
- **Speed:** outward waves only; whiskers are the 95 % CI.
- **Outward order:** Kendall τ of arrival vs distance; above 0 means arrival gets later farther out.
- **Response reach:** unbroken run of rings, from 25 µm, where ≥ 50 % of the measured wedges respond.

In [ ]:
groups = list(dict.fromkeys(s["group"] for s in STACKS_TO_RUN))
names_by_group = {g: [n for n, r in results.items() if r["cfg"]["group"] == g] for g in groups}
order = [n for g in groups for n in names_by_group[g]]
short = {n: n.replace(" (wound right)", "") for n in order}
MARKERS = ["o", "s", "^", "D"]
SPEED_TOP = 30.0          # speed axis limit; an open-ended CI is drawn to the top with an arrow
Y_TOP = 120.0             # arrival-time axis limit (s); later median arrivals are drawn at the top edge with their value
INK, MUTED = "0.15", "0.45"

fig = plt.figure(figsize=(18, 11), layout="constrained")
fig_top, fig_bot = fig.subfigures(2, 1, height_ratios=[1.05, 1], hspace=0.04)

# ---------- top: arrival vs distance, one panel per group ----------
top = np.atleast_1d(fig_top.subplots(1, len(groups), sharey=True))
y_hi = Y_TOP
for ax, g in zip(top, groups):
    color = GROUP_COLORS.get(g, "0.3")
    n_wave = 0
    for k, name in enumerate(names_by_group[g]):
        r = results[name]
        x = r["offs"] * PIXEL_SIZE_UM
        y_true = r["median_arrival"] * FRAME_INTERVAL_S
        y = np.where(y_true > Y_TOP, Y_TOP, y_true)
        for xo, yo in zip(x[y_true > Y_TOP], y_true[y_true > Y_TOP]):
            ax.annotate(f"{yo:.0f} s ↑", (xo, Y_TOP), xytext=(0, -16), textcoords="offset points", ha="center",
                        fontsize=7.5, color=INK)
        mk = MARKERS[k % len(MARKERS)]
        if r["is_wave"]:
            n_wave += 1
            slope, intercept, x_um = r["fit"]
            ax.plot(x_um, slope * x_um + intercept, "-", color=color, lw=2, zorder=2)
            ax.plot(x, y, mk, color=color, ms=8, mec="white", mew=1.2, ls="none", zorder=3, clip_on=False,
                    label=f"{short[name]}: wave, {r['speed']:.1f} µm/s")
        else:
            ax.plot(x, y, mk, color=color, ms=8, mfc="white", mew=1.6, ls="--", lw=1, zorder=3, clip_on=False,
                    label=f"{short[name]}: no outward wave")
    cfg0 = results[names_by_group[g][0]]["cfg"]
    t_off = cfg0["uncage_end_raw"] / RAW_FRAMES_PER_FRAME * FRAME_INTERVAL_S
    if t_off < y_hi:
        ax.axhline(t_off, color=MUTED, ls=":", lw=1.2, zorder=1)
        ax.text(14, t_off + 1.0, f"uncaging off ({t_off:.0f} s)", ha="left", va="bottom", fontsize=8, color=MUTED)
    else:
        ax.text(205, y_hi * 0.97, f"uncaging on for the whole movie ({t_off:.0f} s)", ha="right", va="top",
                fontsize=8, color=MUTED)
    ax.set_title(f"{g}   ({n_wave} of {len(names_by_group[g])} stacks with an outward wave)", fontsize=11,
                 loc="left", color=INK)
    ax.set_xlim(12, 212)
    ax.set_ylim(0, y_hi)
    ax.set_xticks(OFFSETS[OFFSETS >= FIT_MIN_OFFSET] * PIXEL_SIZE_UM)
    ax.set_xlabel("distance from circle edge (µm)")
    ax.grid(axis="y", color="0.92", lw=0.8)
    ax.legend(fontsize=8.5, loc="upper center", bbox_to_anchor=(0.5, -0.13), frameon=False)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
top[0].set_ylabel("median arrival time (s from movie start)")
fig_top.suptitle(f"Arrival time vs distance, {NORM_LABEL}", fontsize=12, x=0.01, ha="left")

# ---------- bottom: speed / outward order / reach, every stack ----------
xpos, xi, spans = {}, 0.0, []
for g in groups:
    x0 = xi
    for n in names_by_group[g]:
        xpos[n] = xi
        xi += 1
    spans.append((g, x0, xi - 1))
    xi += 0.7
bot = fig_bot.subplots(1, 3)
dx = 0.0


def style_bottom(ax, title, ylabel):
    ax.set_title(title, fontsize=11, loc="left", color=INK, pad=34)
    ax.set_ylabel(ylabel)
    ax.set_xticks([xpos[n] for n in order])
    ax.set_xticklabels([short[n] for n in order], rotation=35, ha="right", fontsize=9)
    ax.set_xlim(-0.6, max(xpos.values()) + 0.6)
    ax.grid(axis="y", color="0.92", lw=0.8)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for g, a, b in spans:   # colored group bar above the plot area, group name in text ink
        ax.plot([a - 0.35, b + 0.35], [1.02, 1.02], transform=ax.get_xaxis_transform(), color=GROUP_COLORS.get(g),
                lw=4, solid_capstyle="butt", clip_on=False)


# speed
ax = bot[0]
for n in order:
    r = results[n]
    c, x = GROUP_COLORS.get(r["cfg"]["group"]), xpos[n]
    if r["is_wave"]:
        hi = r["speed_hi"] if np.isfinite(r["speed_hi"]) else SPEED_TOP
        ax.plot([x - dx, x - dx], [r["speed_lo"], hi], color=c, lw=1.6, zorder=2)
        if not np.isfinite(r["speed_hi"]):
            ax.plot(x - dx, SPEED_TOP, "^", color=c, ms=7, clip_on=False, zorder=3)
        ax.plot(x - dx, r["speed"], "o", color=c, ms=9, mec="white", mew=1.2, zorder=4)
    else:
        ax.text(x - dx, 0.6, "no wave", rotation=90, ha="center", va="bottom", fontsize=8, color=MUTED)
ax.set_ylim(0, SPEED_TOP)
style_bottom(ax, "Wave speed (outward waves only)", "speed (µm/s)")
ax.plot([], [], "o", color="0.4", ms=8, mec="white", label="speed with 95% CI")
ax.plot([], [], "^", color="0.4", ms=7, ls="none", label="CI has no upper limit")
ax.legend(fontsize=8, loc="upper right", frameon=False)

# outward order
ax = bot[1]
ax.axhline(0, color=MUTED, lw=1)
for n in order:
    r = results[n]
    c, x = GROUP_COLORS.get(r["cfg"]["group"]), xpos[n]
    ax.plot(x - dx, r["tau"], "o", color=c, ms=9, mec="white", mew=1.2, zorder=4)
    ax.text(x, -0.52, f"p={r['p']:.3f}" if r["p"] >= 0.001 else "p<0.001", ha="center", va="center", fontsize=7.5,
            color=INK)
ax.set_ylim(-0.6, 1.0)
style_bottom(ax, "Outward order (Kendall τ, p-values along the bottom)", "Kendall τ (distance vs arrival)")
ax.text(0.99, 0.97, "above 0 = later farther out", transform=ax.transAxes, ha="right", va="top", fontsize=8,
        color=MUTED)

# reach
ax = bot[2]
for n in order:
    r = results[n]
    c, x = GROUP_COLORS.get(r["cfg"]["group"]), xpos[n]
    ax.plot(x - dx, r["reach_um"], "o", color=c, ms=9, mec="white", mew=1.2, zorder=4)
ax.set_ylim(-8, 215)
style_bottom(ax, "Response reach", "reach (µm)")

for ax in bot:
    for g, a, b in spans:
        ax.text((a + b) / 2, 1.05, g, transform=ax.get_xaxis_transform(), ha="center", va="bottom", fontsize=8.5,
                color=INK)
save_figure(fig, OUT_ROOT / "figures", "all_stacks_one_figure")
plt.show()

## Results

Full circle (suramin 2 and 3: left half); sheet pixels only.

| Group | Stack | Outward wave | Speed (µm/s, 95 % CI) | Immediate response | Response reach | Kendall τ (p) |
|---|---|---|---|---|---|---|
| ATP, uncage full | ATP1_2 | yes | 5.96 (4.71-8.36) | 25 µm | 200 µm | 0.74 (< 0.001) |
| ATP, uncage 35 s | ATP3 | yes | 5.63 (4.16-8.52) | 0 µm | 200 µm | 0.56 (< 0.001) |
| | ATP4 | yes | 18.57 (13.40-59.63) | 0 µm | 200 µm | 0.37 (< 0.001) |
| | ATP5 | yes | 7.48 (4.40-18.69) | 0 µm | 150 µm | 0.39 (< 0.001) |
| suramin, uncage full | Suramin1 | no | - | 0 µm | 25 µm | -0.31 (1.0) |
| | suramin 2 | no | - | 0 µm | 0 µm | - |
| | suramin 3 | no | - | 0 µm | 25 µm | - |

- **By group:**
  - ATP, uncage full: 1 of 1 stacks has an outward wave, 5.96 µm/s.
  - ATP, uncage 35 s: 3 of 3 have one, mean 10.6 ± 7.0 µm/s (SD).
  - Suramin: 0 of 3.
- **ATP4** spreads over about 150 µm within two 7-s frames (figure 5), so its speed is high and its interval
  wide.
- **Sheet pixels.** In the measured rings (25-200 µm), the sheet covers 85 % (ATP1_2), 75 % (ATP3), 76 % (ATP4)
  and 68 % (ATP5) of the tissue pixels, and 9 %, 0 % and 4 % in Suramin1, suramin 2 and suramin 3: under suramin
  the cells flash, but no sheet grows from the circle.
- **Settings.** The verdicts of every stack are the same with `ACTIVE_RISE` 0.2 or 0.5, `SMOOTH_SIGMA_PX` 1 or 3,
  `LATE_TOL` 2 and `RECRUIT_WINDOW` 2; the speeds of ATP1_2, ATP3 and ATP5 stay within 4.9-9.1 µm/s, and ATP4
  ranges from 10 to 36 µm/s.


## How to read the results and what to adjust

- **Outward wave** needs all of: responses in an unbroken run from 25 µm out to at least
  `MIN_WAVE_RINGS` × 25 µm and at least `MIN_BEYOND_RINGS` rings beyond the immediate response, later arrival at
  larger distance (τ > 0), and p < `ALPHA` against the ring-label permutations. The speed is only reported then.
- **"does not spread beyond the immediate response"**: a region around the source responds at once (within
  `INITIAL_WINDOW_S`), but the response does not travel on from there.
- **"responses reach only X µm"**: cells next to the uncaging spot respond but the signal does not spread
  (for example suramin 3: cells at the circle edge respond, nothing beyond ~25 µm).
- **"arrival not later with distance"**: there is activity in the measured wedges, but it is not ordered outward.
  Usually it comes in from the image edge or the wound side (check figure 2, the full-circle maps). If that
  happens in a control, check `angles` for that stack.
- `MIN_PEAK` (0.2) and `NOISE_K` (4) set what counts as a response, together with the rule that the trace must
  start below half of its peak (`starts_low`). The sheet settings (`SHEET`) set which pixels are measured; figure 5
  shows the result for every stack.
- With 7-s frames, a wave that crosses all rings within 1-2 frames has a wide speed interval. The 95 % CI is in
  `wave_summary_all_stacks.csv`.